In [431]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from pfns.bar_distribution import FullSupportBarDistribution
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from tabpfn_time_series import (
    FeatureTransformer,
    TabPFNMode,
    TabPFNTimeSeriesPredictor,
    TabPFNTSPipeline,
    TimeSeriesDataFrame,
)
from tabpfn_time_series.features import (
    AutoSeasonalFeature,
    CalendarFeature,
    RunningIndexFeature,
)
from tabpfn_time_series.plot import plot_forecast
from tqdm import tqdm

from tfmplayground import NanoTabPFNClassifier, NanoTabPFNRegressor
from tfmplayground.models.nanotabpfn import NanoTabPFNModel


In [432]:
DATAPATH = "/data/PFN/Mouse/"
FILE = "data_v4.csv"
VERSUCHSREIHE = [3, 7, 10, 12, 15]
NUM_SERIES = 4000
NUM_TEST_SERIES = 4000
PREDICTION_LENGTH = 3

## Prepare data

### Read data

In [433]:
df = pd.read_csv(os.path.join(DATAPATH, FILE))

# df = df[df["Versuchsreihe"].isin(VERSUCHSREIHE)].drop(columns=["Versuchsreihe"]).reset_index(drop=True)
df = df[df["Versuchsreihe"].isin(VERSUCHSREIHE)].reset_index(drop=True)

# Filter animals that have less than PREDICTION_LENGTH + 1 measurements
df = df.groupby("IdTier").filter(lambda x: len(x) > PREDICTION_LENGTH + 1).reset_index(drop=True)

# Filter animals that have values that are not possible < 5 or > 60
df = df.groupby("IdTier").filter(lambda x: (x["Gewicht"] > 5).all() and (x["Gewicht"] < 60).all()).reset_index(drop=True)

df["IdTier"] = df["IdTier"].astype("category").cat.codes.astype(int)

# Limit df to NUM_SERIES animals for now
NUM_SERIES = min(NUM_SERIES, df["IdTier"].nunique())
selected_ids = np.random.choice(df["IdTier"].unique(), size=NUM_SERIES, replace=False)
df = df[df["IdTier"].isin(selected_ids)].reset_index(drop=True)

# Ensure it is sorted
df = df.sort_values(["IdTier", "Datum"]).reset_index(drop=True)

# Rename to item_id, timestamp, target
df = df.rename(columns={"IdTier": "item_id", "Datum": "timestamp", "Gewicht": "target"})

In [434]:
# TEMPORARY: Filter out animals with less than X measurements
df = df.groupby("item_id").filter(lambda x: len(x) >= 20).reset_index(drop=True)

# And have atleast one intervention
df = df.groupby("item_id").filter(lambda x: (x["Intervention"] == 1).any()).reset_index(drop=True)

### Preprocess data

In [435]:
# Take last PREDICTION_LENGTH rows as test_df, rest as context_df
NUM_TEST_SERIES = min(NUM_TEST_SERIES, df["item_id"].nunique())
selected_ids = np.random.choice(df["item_id"].unique(), size=NUM_TEST_SERIES, replace=False)
test_df = df[df["item_id"].isin(selected_ids)].groupby("item_id").tail(PREDICTION_LENGTH)
# test_df = df.groupby("item_id").tail(PREDICTION_LENGTH)
context_df = df.drop(test_df.index)
future_df = test_df.drop(columns=["target"]).copy()

# Reset index for all DataFrames
context_df = context_df.reset_index(drop=True)
future_df = future_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Copy item_id as new column "IdTier"
context_df["IdTier"] = context_df["item_id"]
future_df["IdTier"] = future_df["item_id"]
test_df["IdTier"] = test_df["item_id"]

base_columns = ["item_id", "timestamp", "target"]
covariate_columns = ["IdTier", "Age", "Intervention"]

context_df = context_df[base_columns + covariate_columns]
future_df = future_df[base_columns[:2] + covariate_columns]
test_df = test_df[base_columns + covariate_columns]

In [436]:
# Causes of death
causes_of_death = df.groupby("item_id").nth(0).reset_index()[["item_id", "Todesursache"]]

# Therapy types:
# 0: No therapy
# 1: Radiation therapy (Bestrahlungen) only
# 2: Chemotherapy (Chemotherapie) only
# 3: Surgery (Operationen) only
# 4: Combination of therapies
# Get sum of columns "Bestrahlungen", "Chemotherapie", "Operationen" for each item_id
def determine_therapy_type(row):
    if row["Bestrahlungen"] == 0 and row["Chemotherapie"] == 0 and row["Operationen"] == 0:
        return 0
    elif row["Bestrahlungen"] > 0 and row["Chemotherapie"] == 0 and row["Operationen"] == 0:
        return 1
    elif row["Bestrahlungen"] == 0 and row["Chemotherapie"] > 0 and row["Operationen"] == 0:
        return 2
    elif row["Bestrahlungen"] == 0 and row["Chemotherapie"] == 0 and row["Operationen"] > 0:
        return 3
    else:
        return 4

therapy_types = {
    0: "No treatment",
    1: "Radiotherapy",
    2: "Chemotherapy",
    3: "Surgery",
    4: "Radio- and chemotherapy"
}

therapy_type = df.groupby("item_id")[["Bestrahlungen", "Chemotherapie", "Operationen"]].sum().reset_index()
therapy_type["type"] = therapy_type.apply(determine_therapy_type, axis=1)

# Info dataframe Versuchsreihe, causes_of_death and therapy_type
info_df = df.groupby("item_id").nth(0).reset_index()[["item_id", "Versuchsreihe"]].merge(
    causes_of_death, on="item_id"
).merge(
    therapy_type[["item_id", "type"]], on="item_id"
)

In [437]:
context_df = TimeSeriesDataFrame(context_df)
future_df["target"] = np.nan
future_df = TimeSeriesDataFrame(future_df)

In [438]:
selected_features = [
    RunningIndexFeature(),
    # CalendarFeature(),
    # AutoSeasonalFeature(),
]

feature_transformer = FeatureTransformer(selected_features)

train_tsdf, test_tsdf = feature_transformer.transform(context_df, future_df)

train_tsdf = train_tsdf.droplevel("timestamp")
test_tsdf = test_tsdf.droplevel("timestamp")

train_tsdf = train_tsdf[["IdTier", "running_index", "Intervention", "target"]]
test_tsdf = test_tsdf[["IdTier", "running_index", "Intervention", "target"]]

## Analysis

### Functions

In [439]:
def mae(
    preds: np.ndarray,
    target: np.ndarray
):
    if not preds.shape == target.shape:
        raise ValueError(f"Preds and target must have the same shape, but got {preds.shape} and {target.shape}")
    
    if preds.ndim == 1:
        return np.abs(preds - target)
    elif preds.ndim == 2:
        return np.mean(np.abs(preds - target), axis=0)
    else:
        raise ValueError("Must be 1 or 2 dimensional")

def mape(
    preds: np.ndarray,
    target: np.ndarray
):
    if not preds.shape == target.shape:
        raise ValueError(f"Preds and target must have the same shape, but got {preds.shape} and {target.shape}")
    
    if preds.ndim == 1:
        return np.abs((preds - target) / target)
    elif preds.ndim == 2:
        return np.mean(np.abs((preds - target) / target), axis=0)
    else:
        raise ValueError("Must be 1 or 2 dimensional")

def plot_prediction(
    id_tier: int,
    data: pd.DataFrame,
    train_x: np.ndarray,
    train_y: np.ndarray,
    test_x: np.ndarray,
    test_y: np.ndarray,
    preds: np.ndarray
):
    row_selected_mouse = data[data["IdTier"] == id_tier].index[0]

    plt.plot(train_x[row_selected_mouse:, 1], train_y[row_selected_mouse:], label='context', zorder=3)
    plt.plot(test_x[..., 1], preds, color='green', label='pfn')
    for i in range(len(train_x[row_selected_mouse:])):
        if train_x[i, 2] == 1:
            plt.axvline(x=train_x[i, 1], color='red', linestyle='--', alpha=0.5, label='intervention' if i == 0 else None, zorder=1)

    for i in range(len(test_x)):
        if test_x[i, 2] == 1:
            plt.axvline(x=test_x[i, 1], color='red', linestyle='--', alpha=0.5, label='intervention' if i == 0 else None, zorder=1)

    plt.plot(test_x[..., 1], test_y, label='remaining data', zorder=2)
    plt.legend()
    plt.show()
    

In [440]:
info_df

,item_id,Versuchsreihe,Todesursache,type
0,2,3,1,1
1,3,3,1,4
2,5,3,1,2
3,7,3,1,2
4,8,3,1,1
...,...,...,...,...
281,680,15,0,1
282,691,15,0,1
283,692,15,0,1
284,699,15,0,1


In [441]:
def select_similar_mice(
    id_tier: int,
    info_df: pd.DataFrame,
    train_tsdf: TimeSeriesDataFrame,
    test_tsdf: TimeSeriesDataFrame,
    test_df: pd.DataFrame,
):
    versuchsreihe_mouse = info_df[info_df["item_id"] == id_tier]["Versuchsreihe"].values[0]
    type_mouse = info_df[info_df["item_id"] == id_tier]["type"].values[0]

    similar_mice = info_df
    # Select mice from the same Versuchsreihe but not id_tier
    similar_mice = similar_mice[(similar_mice["Versuchsreihe"] == versuchsreihe_mouse) & (similar_mice["item_id"] != id_tier)]
    # Select mice with the same therapy type as the main mouse
    similar_mice = similar_mice[similar_mice["type"] == type_mouse]
    
    similar_mice = similar_mice["item_id"].unique()
    similar_mice = np.random.choice(similar_mice, size=min(30, len(similar_mice)), replace=False)

    # similar_mice dataframe contains all data for the selected mice from train_tsdf, test_tsdf and test_df
    similar_mice_train_df = train_tsdf[train_tsdf["IdTier"].isin(similar_mice)]
    similar_mice_test_tsdf = test_tsdf[test_tsdf["IdTier"].isin(similar_mice)]

    for mouse in similar_mice:
        similar_mice_test_tsdf.loc[mouse, "target"] = test_df[test_df["item_id"] == mouse]["target"].values

    combined_similar_mice_df = pd.concat([similar_mice_train_df, similar_mice_test_tsdf], ignore_index=True)
    combined_similar_mice_df = combined_similar_mice_df.sort_values(["IdTier", "running_index"]).reset_index(drop=True)
    
    return combined_similar_mice_df

def predict_mouse(
    id_tier: int,
    reg: NanoTabPFNRegressor,
    info_df: pd.DataFrame,
    train_tsdf: TimeSeriesDataFrame,
    test_tsdf: TimeSeriesDataFrame,
    test_df: pd.DataFrame,
):
    similar_mice = select_similar_mice(
        id_tier=id_tier,
        info_df=info_df,
        train_tsdf=train_tsdf,
        test_tsdf=test_tsdf,
        test_df=test_df
    )

    temp_train_tsdf = pd.concat([similar_mice, train_tsdf[train_tsdf["IdTier"] == id_tier]], ignore_index=True)

    temp_test_tsdf = test_tsdf[test_tsdf["IdTier"] == id_tier]
    temp_test_df = test_df[test_df["IdTier"] == id_tier]

    train_x = temp_train_tsdf[["IdTier", "running_index", "Intervention"]].values
    train_y = temp_train_tsdf["target"].values
    test_x = temp_test_tsdf[["IdTier", "running_index", "Intervention"]].values
    test_y = temp_test_df["target"].values

    # Reset IdTier values
    id_tier_mapping = {id_tier: idx for idx, id_tier in enumerate(temp_train_tsdf["IdTier"].unique())}
    train_x[:, 0] = np.array([id_tier_mapping[id] for id in train_x[:, 0]])
    test_x[:, 0] = np.array([id_tier_mapping[id] for id in test_x[:, 0]])

    with torch.no_grad():
        reg.fit(
            X_train=train_x,
            y_train=train_y,
        )
        preds = reg.predict(
            X_test=test_x,
        )
    
    return preds, test_y

In [442]:
def predict_all_mice(
    reg: NanoTabPFNRegressor,
    therapy_types: dict,
    info_df: pd.DataFrame,
    train_tsdf: TimeSeriesDataFrame,
    test_tsdf: TimeSeriesDataFrame,
    test_df: pd.DataFrame,
):
    num_mice = len(train_tsdf["IdTier"].unique())

    all_preds = np.empty((num_mice, PREDICTION_LENGTH))
    all_targets = np.empty((num_mice, PREDICTION_LENGTH))
    mice_id_therapies = {i: [] for i in therapy_types}
    mice_idx_therapies = {i: [] for i in therapy_types}

    for i, mouse in tqdm(enumerate(train_tsdf["IdTier"].unique())):
        mouse = int(mouse)

        mouse_preds, mouse_targets = predict_mouse(
            id_tier=mouse,
            reg=reg,
            info_df=info_df,
            train_tsdf=train_tsdf,
            test_tsdf=test_tsdf,
            test_df=test_df
        )
        
        all_preds[i] = mouse_preds
        all_targets[i] = mouse_targets
        mice_id_therapies[int(info_df.loc[info_df["item_id"] == mouse, "type"].iloc[0])] += [mouse]
        mice_idx_therapies[int(info_df.loc[info_df["item_id"] == mouse, "type"].iloc[0])] += [i]

    mae_results = np.empty((len(therapy_types.keys()), PREDICTION_LENGTH))
    mape_results = np.empty((len(therapy_types.keys()), PREDICTION_LENGTH))

    for therapy in therapy_types:
        preds = all_preds[mice_idx_therapies[therapy], :]
        targets = all_targets[mice_idx_therapies[therapy], :]
        
        mae_results[therapy] = mae(preds, targets)
        mape_results[therapy] = mape(preds, targets)
    
    return mae_results, mape_results

### Model setup

In [443]:
MODELPATH = "/data/PFN/Mouse/Experiments/Mouse1/"
MODELFILE = "pretrained_mousepfn.pth"
MODELNAME = "Mouse PFN"
BUCKETNAME = "buckets_mousepfn.pth"

In [444]:
# Initialize a classifier
model = NanoTabPFNModel(
    num_attention_heads=6,
    embedding_size=192,
    mlp_hidden_size=768,
    num_layers=6,
    num_outputs=100,
)
model.load_state_dict(
    torch.load(os.path.join(MODELPATH, MODELFILE))
)
bucket_edges = torch.load(os.path.join(MODELPATH, BUCKETNAME))
dist = FullSupportBarDistribution(bucket_edges)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

reg = NanoTabPFNRegressor(
    model=model,
    dist=dist,
    device=device,
)

### Evaluation

In [445]:
# Setup table showing comparison with Eagle eye
table_mae = pd.DataFrame({
    "Model": "Eagle eye",
    "No treatment": np.array([np.nan]),
    "Radiotherapy": np.array([1.041]),
    "Chemotherapy": np.array([0.851]),
    "Surgery": np.array([np.nan]),
    "Radio- and chemotherapy": np.array([1.434]),
    "Overall": np.array([1.1087])
})

table_mape = pd.DataFrame({
    "Model": "Eagle eye (T=1)",
    "No treatment": np.array([np.nan]),
    "Radiotherapy": np.array([0.049]),
    "Chemotherapy": np.array([0.036]),
    "Surgery": np.array([np.nan]),
    "Radio- and chemotherapy": np.array([0.068]),
    "Overall": np.array([0.051])
})

In [446]:
mae_results, mape_results = predict_all_mice(
    reg=reg,
    therapy_types=therapy_types,
    info_df=info_df,
    train_tsdf=train_tsdf,
    test_tsdf=test_tsdf,
    test_df=test_df
)

for i in range(PREDICTION_LENGTH):
    mae_row = {
        "Model": f"{MODELNAME} (T={i + 1})",
        "No treatment": mae_results[0, i],
        "Radiotherapy": mae_results[1, i],
        "Chemotherapy": mae_results[2, i],
        "Surgery": mae_results[3, i],
        "Radio- and chemotherapy": mae_results[4, i],
        "Overall": np.mean(mae_results[[1, 2, 4], i])
    }
    
    mape_row = {
        "Model": f"{MODELNAME} (T={i + 1})",
        "No treatment": mape_results[0, i],
        "Radiotherapy": mape_results[1, i],
        "Chemotherapy": mape_results[2, i],
        "Surgery": mape_results[3, i],
        "Radio- and chemotherapy": mape_results[4, i],
        "Overall": np.mean(mape_results[[1, 2, 4], i])
    }
    
    table_mae.loc[len(table_mae)] = mae_row
    table_mape.loc[len(table_mape)] = mape_row

286it [00:14, 19.99it/s]
/workspaces/nanoPFN/.venv/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/workspaces/nanoPFN/.venv/lib/python3.10/site-packages/numpy/_core/_methods.py:139: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


In [447]:
table_mae.sort_values("Overall")

,Model,No treatment,Radiotherapy,Chemotherapy,Surgery,Radio- and chemotherapy,Overall
1,Mouse PFN (T=1),NaN,0.684777,0.794300,1.031983,0.599450,0.692842
2,Mouse PFN (T=2),NaN,0.890261,0.864484,1.537978,0.726588,0.827111
0,Eagle eye,NaN,1.041000,0.851000,NaN,1.434000,1.108700
3,Mouse PFN (T=3),NaN,1.267776,1.223424,2.319992,1.006172,1.165791


In [448]:
table_mape.sort_values("Overall")

,Model,No treatment,Radiotherapy,Chemotherapy,Surgery,Radio- and chemotherapy,Overall
1,Mouse PFN (T=1),NaN,0.033691,0.032870,0.045225,0.026587,0.031049
2,Mouse PFN (T=2),NaN,0.042125,0.036469,0.069852,0.033672,0.037422
0,Eagle eye (T=1),NaN,0.049000,0.036000,NaN,0.068000,0.051000
3,Mouse PFN (T=3),NaN,0.057212,0.052637,0.105417,0.045764,0.051871
